# Análisis exploratorio — Plantas renovables España 2023-2024

Análisis de datos del sector de energías renovables en España usando
PostgreSQL como fuente de datos y Python para análisis y visualización.

**Dataset:** 6 tablas, ~13.000 registros de producción diaria (2023-2024)  
**Tecnologías:** PostgreSQL · pandas · SQLAlchemy · Plotly  
**Autor:** Juan Jaramillo · 2026

## 1. Configuración e imports

In [ ]:
# Imports y conexión

import sys
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Añadir raíz del proyecto al path
sys.path.insert(0, os.path.abspath('..'))

from src.conexion import get_engine
from src.kpis import (produccion_por_tecnologia, produccion_mensual,
                      ranking_comunidades, anomalias_produccion,
                      kpis_mantenimiento)

engine = get_engine()
print("Conexión establecida correctamente")

## 2. Visión general del dataset

In [ ]:
# Exploración Inicial - Resumen de todas las tablas del dataset
tablas = ['provincias', 'tecnologias', 'plantas_renovables',
          'produccion', 'mantenimiento', 'incidencias']

for tabla in tablas:
    df = pd.read_sql(f"SELECT COUNT(*) AS registros FROM {tabla}", engine)
    print(f"{tabla:25s} → {int(df['registros'].iloc[0]):>6} registros")

In [ ]:
# Vista previa de las plantas con información completa
query = """
    SELECT p.nombre, t.nombre AS tecnologia, pv.comunidad,
           p.potencia_mw, p.inversión_meur, p.num_empleados,
           p.fecha_inicio, p.activa
    FROM plantas_renovables p
    JOIN tecnologias t  ON p.id_tecnologia = t.id_tecnologia
    JOIN provincias pv  ON p.id_provincia  = pv.id_provincia
    ORDER BY p.potencia_mw DESC
"""
df_plantas = pd.read_sql(query, engine)
display(df_plantas)

## 3. Análisis de producción

### 3.1 Producción total por tecnología en 2024

In [ ]:
df_tec = produccion_por_tecnologia(2024)
display(df_tec)

fig = px.bar(df_tec, x='tecnologia', y='total_gwh', color='tecnologia',
             title='Producción total por tecnología — 2024',
             labels={'total_gwh': 'GWh', 'tecnologia': 'Tecnología'},
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(showlegend=False)
fig.show()

### 3.2 Estacionalidad — evolución mensual por tecnología

In [ ]:
df_mensual = produccion_mensual(2024)

fig = px.line(df_mensual, x='mes', y='gwh_mes', color='tecnologia',
              markers=True,
              title='Producción mensual por tecnología — 2024',
              labels={'gwh_mes': 'GWh', 'mes': 'Mes'})
fig.update_xaxes(
    tickvals=list(range(1, 13)),
    ticktext=['Ene','Feb','Mar','Abr','May','Jun',
              'Jul','Ago','Sep','Oct','Nov','Dic']
)
fig.show()

### 3.3 Observación sobre estacionalidad

La energía solar fotovoltaica muestra una marcada estacionalidad,
con producción máxima en verano (junio-agosto) y mínima en invierno.
La energía eólica tiene el patrón inverso: mayor producción en los
meses fríos cuando los vientos son más intensos.
Esta complementariedad entre tecnologías es un factor clave en la
planificación del mix energético renovable.

## 4. Análisis de eficiencia

¿Qué plantas generan más energía por MW instalado?
Este indicador es independiente del tamaño de la planta y permite
comparar la eficiencia operacional real.

In [ ]:
query_eficiencia = """
    WITH produccion_total AS (
        SELECT id_planta,
               ROUND(SUM(kwh_producidos)::NUMERIC / 1000000, 2) AS total_gwh
        FROM produccion
        WHERE EXTRACT(YEAR FROM fecha) = 2024
        GROUP BY id_planta
    )
    SELECT p.nombre,
           t.nombre AS tecnologia,
           pv.comunidad,
           p.potencia_mw,
           pt.total_gwh,
           ROUND((pt.total_gwh / p.potencia_mw * 1000)::NUMERIC, 1) AS kwh_por_kw,
           RANK() OVER (ORDER BY (pt.total_gwh / p.potencia_mw) DESC) AS ranking
    FROM plantas_renovables p
    JOIN tecnologias t         ON p.id_tecnologia = t.id_tecnologia
    JOIN provincias pv         ON p.id_provincia  = pv.id_provincia
    JOIN produccion_total pt   ON p.id_planta     = pt.id_planta
    ORDER BY ranking
"""
df_efic = pd.read_sql(query_eficiencia, engine)
display(df_efic)

fig = px.bar(df_efic, x='nombre', y='kwh_por_kw', color='tecnologia',
             title='Eficiencia por planta — kWh producidos por kW instalado (2024)',
             labels={'kwh_por_kw': 'kWh / kW instalado', 'nombre': 'Planta'})
fig.update_xaxes(tickangle=45)
fig.show()

## 5. Mantenimiento e incidencias

In [ ]:
df_mant = kpis_mantenimiento()
display(df_mant)

fig = px.scatter(
    df_mant,
    x='coste_mantenimiento_eur',
    y='horas_perdidas',
    size='num_incidencias',
    color='tecnologia',
    hover_name='nombre',
    title='Coste de mantenimiento vs horas de parada por planta',
    labels={
        'coste_mantenimiento_eur': 'Coste mantenimiento (€)',
        'horas_perdidas': 'Horas de parada',
        'tecnologia': 'Tecnología'
    }
)
fig.show()

## Conclusiones

1. **La energía solar fotovoltaica** es la tecnología más productiva en términos
   absolutos, gracias a la alta irradiación de las provincias del sur.

2. **La energía eólica** muestra mayor regularidad a lo largo del año,
   siendo especialmente relevante en los meses de invierno.

3. **Andalucía y Extremadura** lideran el ranking de comunidades por producción
   total, beneficiadas por sus condiciones climatológicas.

4. **Las plantas termosolares** destacan en el indicador de eficiencia (kWh/kW),
   gracias a su capacidad de almacenamiento de energía térmica.

5. **El coste de mantenimiento correctivo** supera ampliamente al preventivo
   en las plantas con incidencias críticas, lo que refuerza la importancia
   de los programas de mantenimiento preventivo.